# Gemini edge-labeling demo

This notebook demonstrates a small end-to-end evaluation loop for edge labeling in an agent trajectory.

1. Loads one reconstructed trajectory log and the corresponding human-provided edge labels.
2. Reconstructs source / target text pairs for all five relation families.
3. Samples two random labeled relations from the task.
4. Sends those pairs to Gemini, asks for a label from the allowed label list, and compares the model prediction to the ground-truth label.


## 1. Load one task and parse the reconstructed trajectory log

This cell sets the task file, loads the reconstructed log, and parses it into a dictionary keyed by iteration.

Each parsed iteration contains three text fields:

- `thought`
- `action`
- `result`

These fields are what we later feed into the LLM when constructing relation pairs.


In [24]:
import pandas as pd
from pathlib import Path
import random

TASK_FILE = "django__django-11099.txt"
TASK_STEM = Path(TASK_FILE).stem

LOG_PATH = Path("reconstructed_autocoderover") / TASK_FILE
CSV_ROOT = Path("autocoderover_csv")

RELATION_FAMILIES = [
    "thought_action",
    "thought_thought",
    "action_action",
    "result_thought",
    "result_action",
]

random.seed()


def parse_reconstructed_log(txt_path: Path):
    import re

    text = txt_path.read_text(encoding="utf-8", errors="replace")
    parts = re.split(r"\bIteration\s+(\d+)\b", text)

    out = {}

    def extract_section(chunk: str, name: str, next_names: list[str]) -> str:
        m = re.search(rf"{name}:\s*(.*)", chunk, flags=re.DOTALL)
        if not m:
            return ""
        after = m.group(1)
        cut_positions = []
        for nn in next_names:
            mm = re.search(rf"\n{nn}:\s*", after)
            if mm:
                cut_positions.append(mm.start())
        if cut_positions:
            after = after[: min(cut_positions)]
        return after.strip()

    for idx in range(1, len(parts) - 1, 2):
        it = int(parts[idx])
        chunk = parts[idx + 1].replace("\r\n", "\n")

        thought = extract_section(chunk, "Thought", ["Action", "RESULT", "Result"])
        action = extract_section(chunk, "Action", ["RESULT", "Result", "Thought"])
        result = extract_section(chunk, "Result", ["Thought", "Action"])
        if not result:
            result = extract_section(chunk, "RESULT", ["Thought", "Action"])

        out[it] = {
            "thought": thought,
            "action": action,
            "result": result,
        }

    return out


log_data = parse_reconstructed_log(LOG_PATH)
print(f"Loaded {len(log_data)} iterations from log")

Loaded 4 iterations from log


## 2. Define the label taxonomy and relation-to-text mapping

This cell does two jobs:

- defines the allowed labels for each relation family
- maps a row index `i` to the correct source / target text pair


In [25]:
ALLOWED_LABELS = {
    "thought_action": ["Alignment", "Misalignment"],
    "thought_thought": ["Follow-up", "Refinement", "Redundancy", "Divergence", "Contradiction"],
    "action_action": ["Follow-up", "Refinement", "Repetition", "Divergence"],
    "result_thought": ["Follow-up", "Refinement", "No influence", "Misinterpretation"],
    "result_action": ["Informative", "Triggering", "No influence"],
}


def get_pair_texts(log_data, relation_family, i):
    if relation_family == "thought_action":
        return {
            "source_type": "thought",
            "target_type": "action",
            "source_text": log_data.get(i, {}).get("thought", ""),
            "target_text": log_data.get(i, {}).get("action", ""),
        }

    if relation_family == "thought_thought":
        return {
            "source_type": "thought",
            "target_type": "thought",
            "source_text": log_data.get(i, {}).get("thought", ""),
            "target_text": log_data.get(i + 1, {}).get("thought", ""),
        }

    if relation_family == "action_action":
        return {
            "source_type": "action",
            "target_type": "action",
            "source_text": log_data.get(i, {}).get("action", ""),
            "target_text": log_data.get(i + 1, {}).get("action", ""),
        }

    if relation_family == "result_thought":
        return {
            "source_type": "result",
            "target_type": "thought",
            "source_text": log_data.get(i, {}).get("result", ""),
            "target_text": log_data.get(i + 1, {}).get("thought", ""),
        }

    if relation_family == "result_action":
        return {
            "source_type": "result",
            "target_type": "action",
            "source_text": log_data.get(i, {}).get("result", ""),
            "target_text": log_data.get(i + 1, {}).get("action", ""),
        }

    raise ValueError(f"Unknown relation family: {relation_family}")

## 3. Randomly sample two human-labeled relations

This step pulls two random examples from the existing hand-labeled dataset.

Each sampled example already has:
- a relation family
- a source text
- a target text
- a human-assigned label


In [26]:
all_candidates = []

for family in RELATION_FAMILIES:
    label_path = CSV_ROOT / family / f"{TASK_STEM}.csv"
    if not label_path.exists():
        continue

    labels_df = pd.read_csv(label_path)
    assert list(labels_df.columns) == [
        "label"], f"{family} expected one column ['label'], got {list(labels_df.columns)}"

    for i, row in labels_df.iterrows():
        pair = get_pair_texts(log_data, family, i)

        if pair["source_text"].strip() and pair["target_text"].strip():
            all_candidates.append({
                "relation_family": family,
                "row_index": i,
                "actual_label": row["label"],
                "source_type": pair["source_type"],
                "target_type": pair["target_type"],
                "source_text": pair["source_text"],
                "target_text": pair["target_text"],
            })

sample_examples = random.sample(all_candidates, k=min(2, len(all_candidates)))

pd.DataFrame(sample_examples)[
    ["relation_family", "row_index", "actual_label", "source_type", "target_type"]
]

,relation_family,row_index,actual_label,source_type,target_type
0,result_thought,0,Refinement,result,thought
1,action_action,2,Follow-up,action,action


## 4. Sample two random examples

Here we randomly choose two labeled relations from the candidate set.

Because the sampling is done across all relation families, the two examples can come from different relation types.


In [27]:
sample_examples = random.sample(all_candidates, k=min(2, len(all_candidates)))
pd.DataFrame(sample_examples)[["relation_family", "row_index", "actual_label", "source_type", "target_type"]]

,relation_family,row_index,actual_label,source_type,target_type
0,result_action,2,Triggering,result,action
1,thought_action,3,Alignment,thought,action


## 5. Ask Gemini to predict the labels

This cell:

- loads the Gemini API key from `.env`
- initializes the Gemini client
- sends each sampled example to the model
- asks the model to choose exactly one label from the allowed set for that relation family
- compares the prediction against the actual label

The output is stored in a small comparison table so you can quickly inspect matches vs mismatches.


In [28]:
import json
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3-flash-preview"


def label_one_edge_with_gemini(example):
    relation_family = example["relation_family"]
    allowed = ALLOWED_LABELS[relation_family]

    prompt = f"""
You are labeling the semantic relationship between a source node and a target node in an agent trajectory.

Relation family: {relation_family}

Allowed labels:
{allowed}

Instructions:
- Choose exactly ONE label from the allowed labels.
- Return valid JSON with keys:
  - "predicted_label"
  - "reason"

Source ({example['source_type']}):
\"\"\"
{example['source_text']}
\"\"\"

Target ({example['target_type']}):
\"\"\"
{example['target_text']}
\"\"\"
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
    )

    text = response.text.strip()
    text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

    return json.loads(text)


rows = []

for ex in sample_examples:
    pred = label_one_edge_with_gemini(ex)

    rows.append({
        "relation_family": ex["relation_family"],
        "row_index": ex["row_index"],
        "actual_label": ex["actual_label"],
        "predicted_label": pred["predicted_label"],
        "match": pred["predicted_label"] == ex["actual_label"],
        "reason": pred["reason"],
        "source_type": ex["source_type"],
        "target_type": ex["target_type"],
        "source_preview": ex["source_text"][:160],
        "target_preview": ex["target_text"][:160],
    })

comparison_df = pd.DataFrame(rows)
comparison_df

,relation_family,row_index,actual_label,predicted_label,match,reason,source_type,target_type,source_preview,target_preview
0,result_action,2,Triggering,Triggering,True,The result indicates that the previous reasoni...,result,action,The result is contained in the reasoning\n\n--...,write_patch
1,thought_action,3,Alignment,Alignment,True,The thought contains a detailed set of code mo...,thought,action,# modification 1\n```python\n<file>django/cont...,write_patch


## 6. Print a readable side-by-side comparison

This final cell prints each sampled example:

- relation family
- source / target preview
- actual label
- predicted label
- whether they match
- the model's short rationale


In [29]:
for row in rows:
    print("=" * 100)
    print(f"RELATION FAMILY: {row['relation_family']}")
    print(f"ROW INDEX: {row['row_index']}")
    print(f"ACTUAL LABEL: {row['actual_label']}")
    print(f"PREDICTED LABEL: {row['predicted_label']}")
    print(f"MATCH: {row['match']}")
    print(f"REASON: {row['reason']}")
    print("-" * 100)
    print(f"SOURCE ({row['source_type']}):")
    print(row["source_preview"])
    print("-" * 100)
    print(f"TARGET ({row['target_type']}):")
    print(row["target_preview"])
    print()

RELATION FAMILY: result_action
ROW INDEX: 2
ACTUAL LABEL: Triggering
PREDICTED LABEL: Triggering
MATCH: True
REASON: The result indicates that the previous reasoning step has concluded and determined a course of action, which directly initiates the subsequent 'write_patch' action to implement the fix.
----------------------------------------------------------------------------------------------------
SOURCE (result):
The result is contained in the reasoning

----------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------
TARGET (action):
write_patch

RELATION FAMILY: thought_action
ROW INDEX: 3
ACTUAL LABEL: Alignment
PREDICTED LABEL: Alignment
MATCH: True
REASON: The thought contains a detailed set of code modifications intended to be applied as a patch, and the target action 'write_patch' is the correct and expected step to implement those modifications.
-----